In [1]:
import zipfile
import numpy as np
import math
import pickle
import json
from scipy.sparse import load_npz

In [8]:
input_dir = "../matrices/sparse_matrix_train.npz"
output_file_path = "resultado_baseline.csv"
test_zip_path = "../datos/spotify_test_playlists.zip"
results = {}

In [9]:
train_matrix = load_npz(input_dir)

# Queremos saber las canciones más populares, así que sumamos el valor total de cada columna

In [10]:
track_count = train_matrix.sum(axis=0) # sumamos por columna

track_count = np.asarray(track_count).flatten() # lo transformamos a 1D, que antes estaba como (1, 2262292)

### Ordenamos de más popular a menos, pero lo que nos interesa no es el valor, sino el índice para poder mapearlo, por eso usamos argsort

In [11]:
sorted_indices = np.argsort(track_count)[::-1] # el -500: es para hacer los últimos 500 y el ::-1 es para ordenarlo al revés

In [ ]:
# valores = np.sort(track_count)[-10:][::-1] # por si quisiéramos ver el valor, el número de veces que se guardó esa canción

# Cargamos el mapa de tracks para relacionar el índice con track_uri

In [13]:
with open("../mapas/tracks_train.pkl", "rb") as f:
    track_to_index = pickle.load(f)

In [14]:
index_to_track = {v: k for k, v in track_to_index.items()}

In [15]:
top_n_indices = sorted_indices[:1000]
top_n_uris = [index_to_track[idx] for idx in top_n_indices]

In [16]:
with zipfile.ZipFile(test_zip_path, "r") as zipf:
    for filename in zipf.namelist():
        if filename.endswith("test_input_playlists.json"):
            with zipf.open(filename) as f:
                test_data = json.loads(f.read())
            break

In [18]:
playlists = test_data.get("playlists", [])
total = len(playlists)
print(f"Número de playlists: {total}")

Número de playlists: 10000


In [19]:
for i, playlist in enumerate(playlists):
    pid = playlist["pid"]
    
    # Obtenemos las canciones que YA están en la playlist (semillas)
    # Nota: Dependiendo del formato de test, 'tracks' puede ser una lista de objetos o URIs.
    # Asumimos formato estándar MPD donde tracks es una lista de objetos con "track_uri"
    current_tracks = set()
    for t in playlist["tracks"]:
        current_tracks.add(t["track_uri"])
        
    recommendations = []
    
    # Recorremos nuestra lista de populares
    for track_uri in top_n_uris:
        # Si no tiene la cancion ya, la recomendamos
        if track_uri not in current_tracks:
            recommendations.append(track_uri)
        
        if len(recommendations) == 500:
            break
    
    # Guardamos el resultado para este PID
    results[pid] = recommendations
    
    if (i + 1) % 1000 == 0:
        print(f"Procesado: {i + 1}/{total}")

Procesado: 1000/10000
Procesado: 2000/10000
Procesado: 3000/10000
Procesado: 4000/10000
Procesado: 5000/10000
Procesado: 6000/10000
Procesado: 7000/10000
Procesado: 8000/10000
Procesado: 9000/10000
Procesado: 10000/10000


In [20]:
print(f"Guardando resultados en {output_file_path}...")
with open(output_file_path, "w") as f:
    f.write("team_info, Pablo Fernandez Rubal - Noura el Morchid - Gonzalo Ponte Rodriguez, pablo.fernandez.rubal@udc.es - n.elmorchid@udc.es - g.ponte@udc.es\n")
    for pid, tracks in results.items():
        linea = f"{pid}," + ",".join(tracks)
        f.write("\n" + linea + "\n")
        
print("Archivo generado correctamente")

Guardando resultados en resultado_baseline.csv...
Archivo generado correctamente
